# Module 2 - Detecting Epistemic Enclaves

**Time:** 11:15-12:45  
**Tool focus:** CDlib

This notebook compares several community-detection algorithms and then evaluates the resulting clusters as potential epistemic enclaves.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / ".cache"
PLOTS_DIR = CACHE_DIR / "plots"
(CACHE_DIR / "matplotlib").mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from cdlib import algorithms, evaluation, viz
from IPython.display import Image, display


In [ ]:
def build_demo_graph(seed=42):
    rng = np.random.default_rng(seed)
    communities = [
        {"label": "left-mainstream", "camp": "left", "size": 36, "mean_opinion": -0.55, "enclave": 0},
        {"label": "left-enclave", "camp": "left", "size": 18, "mean_opinion": -0.88, "enclave": 1},
        {"label": "right-mainstream", "camp": "right", "size": 36, "mean_opinion": 0.55, "enclave": 0},
        {"label": "right-enclave", "camp": "right", "size": 18, "mean_opinion": 0.88, "enclave": 1},
    ]
    sizes = [community["size"] for community in communities]
    probability_matrix = [
        [0.18, 0.08, 0.02, 0.004],
        [0.08, 0.25, 0.006, 0.001],
        [0.02, 0.006, 0.18, 0.08],
        [0.004, 0.001, 0.08, 0.25],
    ]

    graph = nx.stochastic_block_model(sizes, probability_matrix, seed=seed)
    graph.graph.clear()

    node_id = 0
    for community in communities:
        for _ in range(community["size"]):
            graph.nodes[node_id]["label"] = f"user_{node_id:03d}"
            graph.nodes[node_id]["community_label"] = community["label"]
            graph.nodes[node_id]["camp"] = community["camp"]
            graph.nodes[node_id]["opinion"] = float(np.clip(rng.normal(community["mean_opinion"], 0.09), -1.0, 1.0))
            graph.nodes[node_id]["enclave"] = int(community["enclave"])
            graph.nodes[node_id]["activity"] = int(rng.poisson(9 if community["enclave"] else 6) + 1)
            node_id += 1

    return graph


def write_demo_graph_files(seed=42):
    DATA_RAW.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    graph = build_demo_graph(seed=seed)

    node_frame = pd.DataFrame(
        [{"node_id": node, **attrs} for node, attrs in graph.nodes(data=True)]
    )
    edge_frame = pd.DataFrame(
        [{"source": source, "target": target} for source, target in graph.edges()]
    )

    nx.write_graphml(graph, DATA_RAW / "workshop_network.graphml")
    nx.write_gexf(graph, DATA_RAW / "workshop_network.gexf")
    nx.write_edgelist(graph, DATA_RAW / "workshop_network.edgelist", data=False)
    node_frame.to_csv(DATA_RAW / "workshop_nodes.csv", index=False)
    edge_frame.to_csv(DATA_PROCESSED / "workshop_edges.csv", index=False)


def load_graph(path):
    path = Path(path)
    if path.suffix == ".graphml":
        graph = nx.read_graphml(path)
    elif path.suffix == ".gexf":
        graph = nx.read_gexf(path)
    else:
        graph = nx.read_edgelist(path, nodetype=int)
        node_frame = pd.read_csv(DATA_RAW / "workshop_nodes.csv")
        attributes = node_frame.set_index("node_id").to_dict(orient="index")
        nx.set_node_attributes(graph, attributes)

    graph = nx.convert_node_labels_to_integers(graph, label_attribute="original_id")
    for _, attrs in graph.nodes(data=True):
        attrs["opinion"] = float(attrs["opinion"])
        attrs["activity"] = int(attrs["activity"])
        attrs["enclave"] = int(attrs["enclave"])
    return graph


In [ ]:
write_demo_graph_files()
G = load_graph(DATA_RAW / "workshop_network.graphml")


## 1. Run several community-detection algorithms


In [ ]:
louvain = algorithms.louvain(G)
leiden = algorithms.leiden(G)
infomap = algorithms.infomap(G)
label_prop = algorithms.label_propagation(G)

clusterings = [louvain, leiden, infomap, label_prop]
pd.DataFrame(
    [{"algorithm": clustering.method_name, "communities": len(clustering.communities)} for clustering in clusterings]
)


In [ ]:
comparison_rows = []
reference = louvain
for clustering in clusterings:
    ari = evaluation.adjusted_rand_index(reference, clustering).score
    nmi = evaluation.normalized_mutual_information(reference, clustering).score
    comparison_rows.append(
        {
            "algorithm": clustering.method_name,
            "ARI_vs_louvain": round(ari, 4),
            "NMI_vs_louvain": round(nmi, 4),
        }
    )

pd.DataFrame(comparison_rows)


In [ ]:
cluster_grid = viz.plot_sim_matrix(clusterings, evaluation.adjusted_rand_index)
matrix_path = PLOTS_DIR / "module2_similarity_matrix.png"
cluster_grid.fig.savefig(matrix_path, bbox_inches="tight")
display(Image(filename=str(matrix_path)))
plt.close("all")


## 2. Visualize the resulting partitions


In [ ]:
position = nx.spring_layout(G, seed=7)
viz.plot_network_clusters(G, louvain, position=position, figsize=(8, 8), plot_labels=False)
cluster_plot_path = PLOTS_DIR / "module2_louvain_network.png"
plt.savefig(cluster_plot_path, bbox_inches="tight")
display(Image(filename=str(cluster_plot_path)))
plt.close("all")


## 3. Evaluate enclave-like structure with CDlib fitness scores


In [ ]:
fitness_rows = []
for clustering in clusterings:
    fitness_rows.append(
        {
            "algorithm": clustering.method_name,
            "modularity": round(evaluation.newman_girvan_modularity(G, clustering).score, 4),
            "internal_edge_density": round(evaluation.internal_edge_density(G, clustering).score, 4),
            "conductance": round(evaluation.conductance(G, clustering).score, 4),
            "avg_distance": round(evaluation.avg_distance(G, clustering).score, 4),
            "hub_dominance": round(evaluation.hub_dominance(G, clustering).score, 4),
        }
    )

fitness_frame = pd.DataFrame(fitness_rows)
fitness_frame


In [ ]:
ax = viz.plot_com_stat(clusterings, evaluation.conductance)
conductance_path = PLOTS_DIR / "module2_conductance.png"
ax.figure.savefig(conductance_path, bbox_inches="tight")
display(Image(filename=str(conductance_path)))
plt.close("all")


## 4. Global and local structural interpretation


In [ ]:
camp_assortativity = nx.attribute_assortativity_coefficient(G, "camp")
enclave_assortativity = nx.attribute_assortativity_coefficient(G, "enclave")

pd.Series(
    {
        "camp_assortativity": round(camp_assortativity, 4),
        "enclave_assortativity": round(enclave_assortativity, 4),
    }
)


In [ ]:
node_to_louvain = {}
for cluster_id, community in enumerate(louvain.communities):
    for node in community:
        node_to_louvain[int(node)] = cluster_id

conformity_rows = []
for node in G.nodes():
    neighbors = list(G.neighbors(node))
    same_neighbor_share = np.nan
    if neighbors:
        same_neighbor_share = np.mean(
            [G.nodes[neighbor]["camp"] == G.nodes[node]["camp"] for neighbor in neighbors]
        )
    conformity_rows.append(
        {
            "node_id": node,
            "camp": G.nodes[node]["camp"],
            "community_label": G.nodes[node]["community_label"],
            "louvain_cluster": node_to_louvain[node],
            "same_neighbor_share": same_neighbor_share,
        }
    )

conformity = pd.DataFrame(conformity_rows)
conformity.groupby("camp")["same_neighbor_share"].describe().round(3)
